Perform lasso, ridge, and elastic net regression techniques to analyze a dataset. 
Compare the effects of different lambda parameters on model outcomes.
Analyze how different regularization approaches affect model performance across various metrics.
Design an optimal modeling strategy that selects between lasso, ridge, or elastic net regression techniques based on dataset characteristics.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet, RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

1. Data Preparation and Feature Scaling
Regularization methods penalize the size of regression coefficients. Because of this, all continuous features must be scaled (z-score normalization) before modeling so that features with larger numeric ranges aren't unfairly penalized.

In [4]:
# 1. Load data and extract relevant columns based on the file layout
df = pd.read_csv("medical_insurance.csv")

continuous_features = ["age", "income", "bmi", "visits_last_year", "risk_score", "claims_count"]
categorical_features = ["sex", "region", "urban_rural", "smoker", "plan_type"]
target = "annual_medical_cost"

# Clean missing values
df_clean = df.dropna(subset=continuous_features + categorical_features + [target])

# Separate target and variables
X_raw = df_clean[continuous_features + categorical_features]
y = df_clean[target]

# One-hot encode categorical features
X_encoded = pd.get_dummies(X_raw, columns=categorical_features, drop_first=True).astype(float)

# Split into Train and Test splits
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# Scale features (Critical step for Regularization!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame just to retain feature tracking names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_encoded.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_encoded.columns)

2. Compare the Effects of Different Lambda ($\alpha$) Parameters

In scikit-learn, the regularization strength parameter $\lambda$ is denoted as alpha.
- A small alpha approaches normal OLS linear regression.
- A massive alpha forces coefficients toward zero, shrinking feature influence.

The code below loops over a range of alphas to showcase how coefficients shrink.

In [5]:
# Array of alpha (lambda) values to test
alphas = [0.1, 1.0, 10.0, 100.0, 1000.0]

ridge_coefs = []
lasso_coefs = []

for a in alphas:
    # Track Ridge shrinking
    ridge = Ridge(alpha=a)
    ridge.fit(X_train_scaled, y_train)
    ridge_coefs.append(ridge.coef_)
    
    # Track Lasso elimination
    lasso = Lasso(alpha=a, max_iter=10000)
    lasso.fit(X_train_scaled, y_train)
    lasso_coefs.append(lasso.coef_)

# Convert to dataframe to see the comparison of a specific feature (e.g., 'age' or 'bmi')
print("--- Ridge Coefficients Shrinkage Profile ---")
ridge_df = pd.DataFrame(ridge_coefs, index=alphas, columns=X_encoded.columns)
print(ridge_df.iloc[:, :4]) # Displaying first 4 features for brevity

print("\n--- Lasso Coefficients Elimination Profile (Features hit exact zero) ---")
lasso_df = pd.DataFrame(lasso_coefs, index=alphas, columns=X_encoded.columns)
print(lasso_df.iloc[:, :4])

--- Ridge Coefficients Shrinkage Profile ---
               age     income        bmi  visits_last_year
0.1    -416.781634 -19.142754  17.717064        252.336445
1.0    -416.740570 -19.142373  17.722450        252.339201
10.0   -416.330283 -19.138558  17.776256        252.366716
100.0  -412.262745 -19.100568  18.308904        252.636704
1000.0 -374.814735 -18.735288  23.142076        254.874567

--- Lasso Coefficients Elimination Profile (Features hit exact zero) ---
               age     income        bmi  visits_last_year
0.1    -416.366237 -19.040338  17.664317        252.342722
1.0    -412.592731 -18.118229  17.193992        252.399575
10.0   -382.141782  -9.011436  11.328150        251.771975
100.0   -76.483696  -0.000000   0.000000        243.273435
1000.0    0.000000   0.000000   0.000000          0.000000


3. Train Regularized Models & Performance Metrics Comparison

We will now use cross-validation (CV variants) to find the absolute mathematically best alpha for Ridge, Lasso, and Elastic Net, and compare them across $R^2$, RMSE, and MAE.

In [6]:
# Array of fine-tuned alpha options to test via Cross Validation
alpha_range = np.logspace(-3, 3, 100)

# 1. Ridge Regression (L2 penalty)
ridge_cv = RidgeCV(alphas=alpha_range, cv=5)
ridge_cv.fit(X_train_scaled, y_train)
best_ridge = Ridge(alpha=ridge_cv.alpha_)
best_ridge.fit(X_train_scaled, y_train)

# 2. Lasso Regression (L1 penalty - feature selection)
lasso_cv = LassoCV(alphas=alpha_range, cv=5, max_iter=10000)
lasso_cv.fit(X_train_scaled, y_train)
best_lasso = Lasso(alpha=lasso_cv.alpha_, max_iter=10000)
best_lasso.fit(X_train_scaled, y_train)

# 3. Elastic Net Regression (Combined L1 and L2 penalties)
# l1_ratio defines the balance. 0.5 means 50% Lasso, 50% Ridge penalty.
elastic_cv = ElasticNetCV(alphas=alpha_range, l1_ratio=[.1, .5, .7, .9, .95, .99, 1], cv=5, max_iter=10000)
elastic_cv.fit(X_train_scaled, y_train)
best_elastic = ElasticNet(alpha=elastic_cv.alpha_, l1_ratio=elastic_cv.l1_ratio_, max_iter=10000)
best_elastic.fit(X_train_scaled, y_train)

# Metrics evaluation collector
models = {
    "Ridge": best_ridge,
    "Lasso": best_lasso,
    "Elastic Net": best_elastic
}

metrics_summary = []

for name, model in models.items():
    preds = model.predict(X_test_scaled)
    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    
    # Count how many features were kept active (coefficient != 0)
    active_features = np.sum(model.coef_ != 0)
    
    metrics_summary.append({
        "Model": name,
        "Best Lambda (Alpha)": getattr(model, "alpha_", "N/A"),
        "Test R²": r2,
        "Test RMSE": rmse,
        "Test MAE": mae,
        "Retained Features": f"{active_features}/{X_train_scaled.shape[1]}"
    })

performance_df = pd.DataFrame(metrics_summary)
print("\n--- Regularization Model Comparison ---")
print(performance_df.to_string(index=False))


--- Regularization Model Comparison ---
      Model Best Lambda (Alpha)  Test R²   Test RMSE    Test MAE Retained Features
      Ridge                 N/A 0.116167 2949.075627 1837.646984             19/19
      Lasso                 N/A 0.116316 2948.827855 1837.303524             14/19
Elastic Net                 N/A 0.116316 2948.827855 1837.303524             14/19


4. Design an Optimal Regularization Selection Strategy
This pipeline functions as an automated decision brain. It maps out characteristics such as:

- Sparsity / Irrelevant Features: If many features are zeroed out by Lasso without killing predictive capacity, it flags it as a sparse system.

- Predictive Performance: It compares error rates (RMSE) across models to determine the framework that generalizes cleanest.

In [7]:
def select_optimal_regularization_strategy(perf_dataframe, models_dict):
    """
    Automated decision tree that selects the ideal regularization engine
    based on model performance metrics and structural data behavior.
    """
    print("\n================ DESIGN STRATEGY ANALYSIS ================")
    
    # 1. Locate model with the lowest test error
    best_perf_row = perf_dataframe.loc[perf_dataframe['Test RMSE'].idxmin()]
    winning_model_name = best_perf_row['Model']
    
    # 2. Check structural parameters (Features zeroed out)
    lasso_active_feats = np.sum(models_dict["Lasso"].coef_ != 0)
    total_feats = models_dict["Lasso"].coef_.shape[0]
    zeroed_out = total_feats - lasso_active_feats
    
    print(f"• Dataset structural note: Lasso eliminated {zeroed_out} out of {total_feats} feature components.")
    print(f"• Top predictive modeling outcome: {winning_model_name} yielded lowest Test RMSE.")
    
    # Execution Rules Strategy Block
    if zeroed_out > 0 and winning_model_name in ["Lasso", "Elastic Net"]:
        strategy_reasoning = (
            f"The dataset benefits from feature selection due to noise/multicollinearity. "
            f"Choosing {winning_model_name} handles these noisy/unneeded dimensions effectively."
        )
    elif winning_model_name == "Ridge":
        strategy_reasoning = (
            "Most parameters hold minor to major predictive distribution weights. "
            "An L2 (Ridge) strategy is preferred because reducing coefficient sizes yields better accuracy than outright zeroing them out."
        )
    else:
        strategy_reasoning = f"Balanced trade-offs indicate {winning_model_name} handles the structural complexity best."
        
    print(f"\n🚀 STRATEGY RECOMMENDATION: Adopt **{winning_model_name}**")
    print(f"REASONING: {strategy_reasoning}")

# Execute the strategic engine
select_optimal_regularization_strategy(performance_df, models)


================ DESIGN STRATEGY ANALYSIS ================
• Dataset structural note: Lasso eliminated 5 out of 19 feature components.
• Top predictive modeling outcome: Lasso yielded lowest Test RMSE.

🚀 STRATEGY RECOMMENDATION: Adopt **Lasso**
REASONING: The dataset benefits from feature selection due to noise/multicollinearity. Choosing Lasso handles these noisy/unneeded dimensions effectively.
